In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

model = "llama-3.3-70b-versatile"

In [ ]:
SYSTEM_PROMPT = """
You are a scheduling assistant.

Use your tools when you need real data.

You can:
- Check the user's calendar.
- Search for contact information.
- Send emails.

Use tool results to decide your next action.
You may call multiple tools if needed.

Always end your final response with a JSON summary in this format:

{
    "summary": "...",
    "actions_taken": ["..."]
}

Do not reveal private chain-of-thought reasoning.
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
def check_calendar(date):
    return f"Calendar for {date}: 10am Standup, 2pm Dentist"

def search_contacts(name):

    contacts = {
        "Sarah": "sarah@example.com",
        "Alice": "alice@example.com"
    }

    return contacts.get(
        name,
        f"No contact found for {name}"
    )

def send_email(to, subject, body):
    return f"Email sent to {to}"

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_calendar",
            "description": "Check calendar events for a given date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string"
                    }
                },
                "required": ["date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_contacts",
            "description": "Search for a person's email address by name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string"
                    }
                },
                "required": ["name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to someone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {
                        "type": "string"
                    },
                    "subject": {
                        "type": "string"
                    },
                    "body": {
                        "type": "string"
                    }
                },
                "required": [
                    "to",
                    "subject",
                    "body"
                ]
            }
        }
    }
]

In [ ]:
def check_input(message):

    blocked = [
        "medical",
        "legal",
        "financial advice"
    ]

    for term in blocked:

        if term in message.lower():
            return "I can only help with scheduling, contacts, and email."

    return None

In [ ]:
def execute_tool(name, args):

    if name == "check_calendar":
        return check_calendar(**args)

    elif name == "search_contacts":
        return search_contacts(**args)

    elif name == "send_email":

        print(f"\nProposed email:")
        print(f"To: {args['to']}")
        print(f"Subject: {args['subject']}")
        print(f"Body: {args['body']}")

        confirm = input(
            "\nSend this email? (y/n): "
        )

        if confirm.lower() != "y":
            return "Email cancelled by user."

        return send_email(**args)

    return f"Unknown tool: {name}"

In [ ]:
def run_agent(messages):

    while True:

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason
        msg = response.choices[0].message

        messages.append(msg)

        # Final answer
        if finish_reason == "stop":
            return msg.content

        # Tool calls
        if finish_reason == "tool_calls":

            for tc in msg.tool_calls:

                name = tc.function.name
                args = json.loads(
                    tc.function.arguments
                )

                print(
                    f"\nAction: {name}({args})"
                )

                result = execute_tool(
                    name,
                    args
                )

                print(
                    f"Observation: {result}"
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })

In [ ]:
def run(user_message):

    # Step 1: Check input guardrail
    guard_result = check_input(
        user_message
    )

    if guard_result:
        return guard_result

    # Step 2: Add user message
    messages.append({
        "role": "user",
        "content": user_message
    })

    # Step 3: Run agent
    return run_agent(messages)

In [ ]:
user_message = """
Email Sarah my calendar summary for today.
"""

answer = run(user_message)

print("\nFinal Answer:")
print(answer)

In [ ]:
answer = run(
    "Give me financial advice about investing."
)
print(answer)